In [ ]:
%load_ext blackcellmagic 
# %%black -l 120
%load_ext autoreload
%autoreload 2

In [ ]:
# %%black -l 120
import os
import jax
import jax.numpy as jnp
from tests.utils import Generator
from slimdqn.algorithms.dqn import DQN
from slimdqn.algorithms.dqnrcshared import DQNRCShared
from slimdqn.algorithms.idqnshared import iDQNShared
from slimdqn.algorithms.gidqnshared import GiDQNShared


# reproducability and determinism
os.environ["XLA_FLAGS"] = (
    # "--xla_gpu_autotune_level=0 "
    # "--xla_gpu_deterministic_ops=true "
    "--xla_backend_optimization_level=0 "
)

batch_size = 16 #and larger batch size of 32, or what the standard is
features = [16, 256] #add second architecture here of Dopamine
observation_dim = (16, 16, 2) #and native down scale of 84 by 84 with 4 stacked
n_actions = 10
n_bellman_iterations = 5


def count_params(params):
    return sum(x.size for x in jax.tree.leaves(params))


def count_flops(q, has_target_params=False):
    best_action_compiled = (
        jax.jit(q.best_action).lower(q.params, sample_generator.state(jax.random.PRNGKey(0))).compile()
    )
    if not has_target_params:
        learn_on_batch_compiled = (
            jax.jit(q.learn_on_batch)
            .lower(q.params, q.optimizer_state, sample_generator.samples(jax.random.PRNGKey(0)), jnp.ones(batch_size))
            .compile()
        )
    else:
        learn_on_batch_compiled = (
            jax.jit(q.learn_on_batch)
            .lower(
                q.params,
                q.target_params,
                q.optimizer_state,
                sample_generator.samples(jax.random.PRNGKey(0)),
                jnp.ones(batch_size),
            )
            .compile()
        )

    return best_action_compiled, learn_on_batch_compiled


sample_generator = Generator(batch_size, observation_dim, n_actions)


print("--- DQN ---")
q_dqn = DQN(jax.random.PRNGKey(0), observation_dim, n_actions, features, 6.25e-5, 0.99, 1, 1, 8000)
q_dqn_best_action_compiled, q_dqn_learn_on_batch_compiled = count_flops(q_dqn, has_target_params=True)

print(count_params(q_dqn.params) + count_params(q_dqn.target_params), "parameters")
print("FLOPs best action: ", q_dqn_best_action_compiled.cost_analysis()[0]["flops"])
print("FLOPs to learn on a batch: ", q_dqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")

print("--- DQNRC ---")
q_qrc = DQNRCShared(jax.random.PRNGKey(0), observation_dim, n_actions, features, 6.25e-5, 0.99, 1, 0.25, 8000, 1)
q_qrc_best_action_compiled, q_qrc_learn_on_batch_compiled = count_flops(q_qrc, has_target_params=False)

print(count_params(q_qrc.params), "parameters")
print("FLOPs best action: ", q_qrc_best_action_compiled.cost_analysis()[0]["flops"])
print("FLOPs to learn on a batch: ", q_qrc_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")

print("--- i-DQN ---")
q_idqn = iDQNShared(jax.random.PRNGKey(0), observation_dim, n_actions, n_bellman_iterations, features, 6.25e-5, 0.99, 1, 0.25, 8000)
q_idqn_best_action_compiled, q_idqn_learn_on_batch_compiled = count_flops(q_idqn, has_target_params=True)

print(count_params(q_idqn.params) + count_params(q_idqn.target_params), "parameters")
print("FLOPs best action: ", q_idqn_best_action_compiled.cost_analysis()[0]["flops"])
print("FLOPs to learn on a batch: ", q_idqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")


print("--- Gi-DQN ---")
q_gidqn = GiDQNShared(jax.random.PRNGKey(0), observation_dim, n_actions, n_bellman_iterations, features, 6.25e-5, 0.99, 1, 0.25, 8000, 1)
q_gidqn_best_action_compiled, q_gidqn_learn_on_batch_compiled = count_flops(q_gidqn, has_target_params=True)

print(count_params(q_gidqn.params) + count_params(q_gidqn.target_params), "parameters")
print("FLOPs best action: ", q_gidqn_best_action_compiled.cost_analysis()[0]["flops"])
print("FLOPs to learn on a batch: ", q_gidqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")

In [ ]:
# %%
batch_size = 16
features = [16, 256]
observation_dim = (16, 16, 2)
n_actions = 10
n_bellman_iterations = 5

def tree_bytes(tree):
    return sum(leaf.nbytes for leaf in jax.tree.leaves(tree))

def human(n):
    for u in ("B", "KiB", "MiB", "GiB", "TiB"):
        if n < 1024:
            return f"{n:7.2f} {u}"
        n /= 1024

metrics = {"flops": {}, "num_params": {}, "memory_bytes": {}}

def report(name, q, has_target_params):
    best, learn = count_flops(q, has_target_params=has_target_params)
    n_params = count_params(q.params) + (count_params(q.target_params) if has_target_params else 0)
    nn_mem = (
        tree_bytes(q.params)
        + tree_bytes(q.optimizer_state)
        + (tree_bytes(q.target_params) if has_target_params else 0)
    )

    metrics["num_params"][name] = n_params
    metrics["flops"][name] = learn.cost_analysis()[0]["flops"]
    metrics["memory_bytes"][name] = nn_mem

    print(f"--- {name} ---")
    print(
        f"  parameters    : {n_params:,}  "
        f"(online {count_params(q.params):,}"
        f"{' + target ' + format(count_params(q.target_params), ',') if has_target_params else ''})"
    )
    print(f"  FLOPs best    : {best.cost_analysis()[0]['flops']:,.0f}")
    print(f"  FLOPs learn   : {learn.cost_analysis()[0]['flops']:,.0f}")
    print(f"  NN memory     : {human(nn_mem)}\n")


report(
    "dqn",
    DQN(jax.random.PRNGKey(0), observation_dim, n_actions, features, 6.25e-5, 0.99, 1, 1, 8000),
    has_target_params=True,
)
report(
    "qrc",
    DQNRCShared(jax.random.PRNGKey(0), observation_dim, n_actions, features, 6.25e-5, 0.99, 1, 0.25, 8000, 1),
    has_target_params=False,
)
report(
    "idqn",
    iDQNShared(jax.random.PRNGKey(0), observation_dim, n_actions, n_bellman_iterations, features, 6.25e-5, 0.99, 1, 0.25, 8000),
    has_target_params=True,
)
report(
    "gidqn",
    GiDQNShared(jax.random.PRNGKey(0), observation_dim, n_actions, n_bellman_iterations, features, 6.25e-5, 0.99, 1, 0.25, 8000, 1),
    has_target_params=True,
)

print(metrics)
import copy

metrics_small = copy.deepcopy(metrics)

Large Scale


In [ ]:
# %%black -l 120
import os
import jax
import jax.numpy as jnp
from tests.utils import Generator
from slimdqn.algorithms.dqn import DQN
from slimdqn.algorithms.dqnrcshared import DQNRCShared
from slimdqn.algorithms.idqnshared import iDQNShared
from slimdqn.algorithms.gidqnshared import GiDQNShared


# reproducability and determinism
os.environ["XLA_FLAGS"] = (
    # "--xla_gpu_autotune_level=0 "
    # "--xla_gpu_deterministic_ops=true "
    "--xla_backend_optimization_level=0 "
)

batch_size = 32 #and larger batch size of 32, or what the standard is
features = [32, 64, 64, 512] #add second architecture here of Dopamine
observation_dim = (84, 84, 4) #and native down scale of 84 by 84 with 4 stacked
n_actions = 10
n_bellman_iterations = 5


def count_params(params):
    return sum(x.size for x in jax.tree.leaves(params))


def count_flops(q, has_target_params=False):
    best_action_compiled = (
        jax.jit(q.best_action).lower(q.params, sample_generator.state(jax.random.PRNGKey(0))).compile()
    )
    if not has_target_params:
        learn_on_batch_compiled = (
            jax.jit(q.learn_on_batch)
            .lower(q.params, q.optimizer_state, sample_generator.samples(jax.random.PRNGKey(0)), jnp.ones(batch_size))
            .compile()
        )
    else:
        learn_on_batch_compiled = (
            jax.jit(q.learn_on_batch)
            .lower(
                q.params,
                q.target_params,
                q.optimizer_state,
                sample_generator.samples(jax.random.PRNGKey(0)),
                jnp.ones(batch_size),
            )
            .compile()
        )

    return best_action_compiled, learn_on_batch_compiled


sample_generator = Generator(batch_size, observation_dim, n_actions)


print("--- DQN ---")
q_dqn = DQN(jax.random.PRNGKey(0), observation_dim, n_actions, features, 6.25e-5, 0.99, 1, 1, 8000)
q_dqn_best_action_compiled, q_dqn_learn_on_batch_compiled = count_flops(q_dqn, has_target_params=True)

print(count_params(q_dqn.params) + count_params(q_dqn.target_params), "parameters")
print("FLOPs best action: ", q_dqn_best_action_compiled.cost_analysis()[0]["flops"])
print("FLOPs to learn on a batch: ", q_dqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")

print("--- DQNRC ---")
q_qrc = DQNRCShared(jax.random.PRNGKey(0), observation_dim, n_actions, features, 6.25e-5, 0.99, 1, 0.25, 8000, 1)
q_qrc_best_action_compiled, q_qrc_learn_on_batch_compiled = count_flops(q_qrc, has_target_params=False)

print(count_params(q_qrc.params), "parameters")
print("FLOPs best action: ", q_qrc_best_action_compiled.cost_analysis()[0]["flops"])
print("FLOPs to learn on a batch: ", q_qrc_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")

print("--- i-DQN ---")
q_idqn = iDQNShared(jax.random.PRNGKey(0), observation_dim, n_actions, n_bellman_iterations, features, 6.25e-5, 0.99, 1, 0.25, 8000)
q_idqn_best_action_compiled, q_idqn_learn_on_batch_compiled = count_flops(q_idqn, has_target_params=True)

print(count_params(q_idqn.params) + count_params(q_idqn.target_params), "parameters")
print("FLOPs best action: ", q_idqn_best_action_compiled.cost_analysis()[0]["flops"])
print("FLOPs to learn on a batch: ", q_idqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")


print("--- Gi-DQN ---")
q_gidqn = GiDQNShared(jax.random.PRNGKey(0), observation_dim, n_actions, n_bellman_iterations, features, 6.25e-5, 0.99, 1, 0.25, 8000, 1)
q_gidqn_best_action_compiled, q_gidqn_learn_on_batch_compiled = count_flops(q_gidqn, has_target_params=True)

print(count_params(q_gidqn.params) + count_params(q_gidqn.target_params), "parameters")
print("FLOPs best action: ", q_gidqn_best_action_compiled.cost_analysis()[0]["flops"])
print("FLOPs to learn on a batch: ", q_gidqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")

In [ ]:
# %%
batch_size = 32
features = [32, 64, 64, 512]
observation_dim = (84, 84, 4)
n_actions = 10
n_bellman_iterations = 5

def tree_bytes(tree):
    return sum(leaf.nbytes for leaf in jax.tree.leaves(tree))

def human(n):
    for u in ("B", "KiB", "MiB", "GiB", "TiB"):
        if n < 1024:
            return f"{n:7.2f} {u}"
        n /= 1024

metrics = {"flops": {}, "num_params": {}, "memory_bytes": {}}

def report(name, q, has_target_params):
    best, learn = count_flops(q, has_target_params=has_target_params)
    n_params = count_params(q.params) + (count_params(q.target_params) if has_target_params else 0)
    nn_mem = (
        tree_bytes(q.params)
        + tree_bytes(q.optimizer_state)
        + (tree_bytes(q.target_params) if has_target_params else 0)
    )

    metrics["num_params"][name] = n_params
    metrics["flops"][name] = learn.cost_analysis()[0]["flops"]
    metrics["memory_bytes"][name] = nn_mem

    print(f"--- {name} ---")
    print(
        f"  parameters    : {n_params:,}  "
        f"(online {count_params(q.params):,}"
        f"{' + target ' + format(count_params(q.target_params), ',') if has_target_params else ''})"
    )
    print(f"  FLOPs best    : {best.cost_analysis()[0]['flops']:,.0f}")
    print(f"  FLOPs learn   : {learn.cost_analysis()[0]['flops']:,.0f}")
    print(f"  NN memory     : {human(nn_mem)}\n")


report(
    "dqn",
    DQN(jax.random.PRNGKey(0), observation_dim, n_actions, features, 6.25e-5, 0.99, 1, 1, 8000),
    has_target_params=True,
)
report(
    "qrc",
    DQNRCShared(jax.random.PRNGKey(0), observation_dim, n_actions, features, 6.25e-5, 0.99, 1, 0.25, 8000, 1),
    has_target_params=False,
)
report(
    "idqn",
    iDQNShared(jax.random.PRNGKey(0), observation_dim, n_actions, n_bellman_iterations, features, 6.25e-5, 0.99, 1, 0.25, 8000),
    has_target_params=True,
)
report(
    "gidqn",
    GiDQNShared(jax.random.PRNGKey(0), observation_dim, n_actions, n_bellman_iterations, features, 6.25e-5, 0.99, 1, 0.25, 8000, 1),
    has_target_params=True,
)

print(metrics)
metrics_large = copy.deepcopy(metrics)

In [ ]:
reduction = {
    metric: {
        name: 1 - (metrics_small[metric][name] / metrics_large[metric][name])
        for name in metrics_small[metric]
    }
    for metric in metrics_small
}

for metric, values in reduction.items():
    print(f"--- {metric} reduction (small vs large) ---")
    for name, r in values.items():
        print(f"  {name:10s}: {r * 100:6.2f}%")
    print()

print(reduction)